# 03 — Price Forecasting Models
**CamAgri Agricultural Risk Assessment Platform**

This notebook trains, evaluates and persists the three price forecasting models:
- **ARIMA / SARIMAX** — Free tier: classical time-series with seasonal differencing
- **Prophet** — Medium tier: additive decomposition with Cameroon-specific seasonality
- **Ensemble (ARIMA + Prophet + XGBoost)** — Premium tier: weighted blending

**Sections**
1. Setup & data loading
2. Stationarity tests & differencing
3. ARIMA / SARIMAX model
4. Prophet model
5. XGBoost regressor on engineered features
6. Ensemble blending
7. Backtesting & evaluation
8. Tier comparison
9. Save models

## 1. Setup & Data Loading

In [ ]:
import warnings, os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
from pathlib import Path
from itertools import product
import joblib

# Stats / ML
from statsmodels.tsa.stattools import adfuller, kpss
from statsmodels.tsa.seasonal import seasonal_decompose
from statsmodels.tsa.statespace.sarimax import SARIMAX
from statsmodels.graphics.tsaplots import plot_acf, plot_pacf
from sklearn.metrics import mean_squared_error, mean_absolute_error
from sklearn.model_selection import TimeSeriesSplit
from sklearn.preprocessing import RobustScaler

try:
    from prophet import Prophet
    PROPHET_AVAILABLE = True
except ImportError:
    PROPHET_AVAILABLE = False
    print('[WARNING] Prophet not installed. Run: pip install prophet')

try:
    import xgboost as xgb
    XGB_AVAILABLE = True
except ImportError:
    XGB_AVAILABLE = False
    print('[WARNING] XGBoost not installed. Run: pip install xgboost')

warnings.filterwarnings('ignore')

DATA_ROOT    = Path('../data')
FEAT_DIR     = Path('../outputs/features')
MODELS_DIR   = Path('../models')
OUTPUT_DIR   = Path('../outputs/forecasting')
PRICES_DIR   = DATA_ROOT / 'prices'
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
MODELS_DIR.mkdir(parents=True, exist_ok=True)

plt.rcParams.update({
    'figure.facecolor':'#0d0d0d','axes.facecolor':'#111111',
    'axes.edgecolor':'#333333','text.color':'#cccccc',
    'axes.labelcolor':'#cccccc','axes.titlecolor':'#ffffff',
    'xtick.color':'#666666','ytick.color':'#666666',
    'grid.color':'#222222','grid.alpha':0.5,'font.size':11,
})
LIME = '#CAFF00'

# Model crop for prototyping
DEMO_CROP = 'maize'
print(f'Setup complete. Demo crop: {DEMO_CROP}')

In [ ]:
# Load engineered price features (from notebook 02)
try:
    price_df = pd.read_parquet(FEAT_DIR / 'price_features.parquet')
    price_df['date'] = pd.to_datetime(price_df['date'])
    print('Loaded engineered features from parquet.')
except FileNotFoundError:
    # Fallback: load raw and compute national average
    frames = [pd.read_csv(f, parse_dates=['date']) for f in sorted(PRICES_DIR.glob('*_price.csv'))]
    raw = pd.concat(frames, ignore_index=True)
    price_df = (
        raw.groupby(['date','crop'])['price_xaf_per_kg']
        .mean().reset_index().sort_values(['crop','date'])
    )
    print('Loaded raw price data (run notebook 02 for full features).')

# Work with a single crop series
series = (
    price_df[price_df.crop == DEMO_CROP]
    .set_index('date')['price_xaf_per_kg']
    .sort_index()
    .dropna()
)
print(f'Series length: {len(series)} weekly observations')
print(f'Date range:    {series.index[0].date()} → {series.index[-1].date()}')

## 2. Stationarity Tests & Differencing

In [ ]:
# ── 2.1 Visual decomposition ──────────────────────────────────────────────
decomp = seasonal_decompose(series, model='multiplicative', period=52, extrapolate_trend='freq')

fig, axes = plt.subplots(4, 1, figsize=(15, 11), sharex=True)
labels = ['Original', 'Trend', 'Seasonal', 'Residual']
colors = [LIME, '#38B6FF', '#FFB800', '#FF6B6B']
for ax, data, label, c in zip(axes,
    [decomp.observed, decomp.trend, decomp.seasonal, decomp.resid],
    labels, colors):
    ax.plot(data.index, data.values, color=c, lw=1.3)
    ax.set_ylabel(label, fontsize=10)
    ax.grid(True, alpha=0.25)
axes[0].set_title(f'{DEMO_CROP.upper()} Price — Multiplicative STL Decomposition')
plt.tight_layout()
plt.savefig(OUTPUT_DIR/'decomposition.png', dpi=140, bbox_inches='tight')
plt.show()

In [ ]:
# ── 2.2 ADF & KPSS stationarity tests ─────────────────────────────────────
def run_stationarity_tests(ts, label='series'):
    print(f'\n=== Stationarity Tests: {label} ===')
    # ADF — H0: unit root (non-stationary)
    adf_stat, adf_p, _, _, adf_cv, _ = adfuller(ts.dropna(), autolag='AIC')
    print(f'ADF  stat={adf_stat:.4f}  p={adf_p:.4f}  '
          f'  → {"STATIONARY" if adf_p < 0.05 else "NON-STATIONARY"} (5%)')
    # KPSS — H0: stationary
    kpss_stat, kpss_p, _, kpss_cv = kpss(ts.dropna(), regression='ct', nlags='auto')
    print(f'KPSS stat={kpss_stat:.4f}  p={kpss_p:.4f}  '
          f'  → {"NON-STATIONARY" if kpss_p < 0.05 else "STATIONARY"} (5%)')
    return adf_p, kpss_p

adf_p, kpss_p = run_stationarity_tests(series, f'{DEMO_CROP} price')
diff1 = series.diff(1).dropna()
run_stationarity_tests(diff1, f'{DEMO_CROP} price (1st diff)')
log_diff = np.log(series).diff(1).dropna()
run_stationarity_tests(log_diff, f'{DEMO_CROP} log-price (1st diff)')

In [ ]:
# ── 2.3 ACF / PACF to identify ARIMA orders ───────────────────────────────
fig, axes = plt.subplots(2, 2, figsize=(16, 9))

plot_acf( series,   lags=60, ax=axes[0,0], title=f'{DEMO_CROP} Price — ACF',  color=LIME)
plot_pacf(series,   lags=60, ax=axes[0,1], title=f'{DEMO_CROP} Price — PACF', color=LIME)
plot_acf( log_diff, lags=60, ax=axes[1,0], title='Log-Diff Price — ACF',  color='#38B6FF')
plot_pacf(log_diff, lags=60, ax=axes[1,1], title='Log-Diff Price — PACF', color='#38B6FF')

for ax in axes.flat:
    ax.set_facecolor('#111111')
    for line in ax.get_lines(): line.set_color(line.get_color().replace('#1f77b4', LIME))

plt.tight_layout()
plt.savefig(OUTPUT_DIR/'acf_pacf.png', dpi=140, bbox_inches='tight')
plt.show()

## 3. ARIMA / SARIMAX Model (Free Tier)

In [ ]:
# ── 3.1 Train/test split ──────────────────────────────────────────────────
FORECAST_HORIZON = 12   # 12 weeks ahead
TEST_SIZE        = 52   # hold out last year

train_series = series.iloc[:-TEST_SIZE]
test_series  = series.iloc[-TEST_SIZE:]

print(f'Train: {len(train_series)} obs  ({train_series.index[0].date()} → {train_series.index[-1].date()})')
print(f'Test : {len(test_series)} obs  ({test_series.index[0].date()}  → {test_series.index[-1].date()})')

In [ ]:
# ── 3.2 SARIMAX fit ───────────────────────────────────────────────────────
# Order selected via ACF/PACF inspection + AIC grid (simplified here)
print('Fitting SARIMAX(2,1,2)x(1,1,1,52)...')
try:
    sarimax_model = SARIMAX(
        train_series,
        order=(2, 1, 2),
        seasonal_order=(1, 1, 1, 52),
        enforce_stationarity=False,
        enforce_invertibility=False,
    )
    sarimax_fit = sarimax_model.fit(disp=False, maxiter=200)
    print(sarimax_fit.summary().tables[0])
except Exception as e:
    # Fallback to simpler order if seasonal fails
    print(f'Seasonal ARIMA failed ({e}), falling back to ARIMA(2,1,2)...')
    sarimax_model = SARIMAX(train_series, order=(2, 1, 2),
                            enforce_stationarity=False, enforce_invertibility=False)
    sarimax_fit = sarimax_model.fit(disp=False, maxiter=200)
    print('ARIMA(2,1,2) fitted.')

In [ ]:
# ── 3.3 SARIMAX forecast & evaluation ─────────────────────────────────────
def evaluate_forecast(y_true, y_pred, model_name='Model'):
    rmse = np.sqrt(mean_squared_error(y_true, y_pred))
    mae  = mean_absolute_error(y_true, y_pred)
    mape = np.mean(np.abs((y_true - y_pred) / (y_true + 1e-9))) * 100
    bias = np.mean(y_pred - y_true) / np.mean(y_true)
    print(f'{model_name:30s}  RMSE={rmse:8.2f}  MAE={mae:7.2f}  MAPE={mape:5.2f}%  Bias={bias:+.4f}')
    return {'rmse': round(rmse,2), 'mae': round(mae,2), 'mape': round(mape,2), 'bias': round(bias,4)}

# One-step rolling forecast on test set
sarimax_preds = []
history = list(train_series)
for t in range(TEST_SIZE):
    model = SARIMAX(history, order=(2,1,2), enforce_stationarity=False, enforce_invertibility=False)
    fit   = model.fit(disp=False, maxiter=100)
    yhat  = fit.forecast(1)[0]
    sarimax_preds.append(yhat)
    history.append(test_series.iloc[t])

sarimax_preds = np.array(sarimax_preds)
arima_metrics = evaluate_forecast(test_series.values, sarimax_preds, 'ARIMA (Free tier)')

In [ ]:
# ── 3.4 ARIMA residual diagnostics ────────────────────────────────────────
fig = sarimax_fit.plot_diagnostics(figsize=(15, 9))
fig.suptitle(f'SARIMAX Residual Diagnostics — {DEMO_CROP}', fontsize=13)
plt.tight_layout()
plt.savefig(OUTPUT_DIR/'arima_diagnostics.png', dpi=140, bbox_inches='tight')
plt.show()

print('\nLjung-Box test (residuals whiteness):')
from statsmodels.stats.diagnostic import acorr_ljungbox
lb = acorr_ljungbox(sarimax_fit.resid, lags=[10,20,52], return_df=True)
print(lb)

## 4. Prophet Model (Medium Tier)

In [ ]:
# ── 4.1 Prepare Prophet data format ─────────────────────────────────────
prophet_train = train_series.reset_index().rename(columns={'date':'ds','price_xaf_per_kg':'y'})
prophet_test  = test_series.reset_index().rename(columns={'date':'ds','price_xaf_per_kg':'y'})
print('Prophet train shape:', prophet_train.shape)
print(prophet_train.head(3))

In [ ]:
# ── 4.2 Prophet model with Cameroon-specific seasonalities ────────────────
if PROPHET_AVAILABLE:
    prophet_model = Prophet(
        yearly_seasonality   = True,
        weekly_seasonality   = False,
        daily_seasonality    = False,
        seasonality_mode     = 'multiplicative',  # crop prices are multiplicative
        changepoint_prior_scale   = 0.15,   # flexibility for price regime changes
        seasonality_prior_scale   = 10.0,
        interval_width       = 0.80,
    )
    # Cameroon-specific: biannual harvest cycle (~26 weeks)
    prophet_model.add_seasonality(name='biannual', period=26, fourier_order=5)
    # Long dry season effect
    prophet_model.add_seasonality(name='quarterly', period=13, fourier_order=3)

    prophet_model.fit(prophet_train)

    # Forecast over test period
    future   = prophet_model.make_future_dataframe(periods=TEST_SIZE, freq='W')
    forecast  = prophet_model.predict(future)

    prophet_preds = forecast.tail(TEST_SIZE)['yhat'].values
    prophet_lower = forecast.tail(TEST_SIZE)['yhat_lower'].values
    prophet_upper = forecast.tail(TEST_SIZE)['yhat_upper'].values

    prophet_metrics = evaluate_forecast(test_series.values, prophet_preds, 'Prophet (Medium tier)')
else:
    print('Prophet unavailable — using ARIMA predictions as placeholder.')
    prophet_preds   = sarimax_preds * 1.02   # placeholder
    prophet_lower   = prophet_preds * 0.90
    prophet_upper   = prophet_preds * 1.10
    prophet_metrics = arima_metrics

In [ ]:
# ── 4.3 Prophet component plots ───────────────────────────────────────────
if PROPHET_AVAILABLE:
    fig = prophet_model.plot_components(forecast)
    fig.suptitle(f'Prophet Components — {DEMO_CROP}', fontsize=13)
    plt.savefig(OUTPUT_DIR/'prophet_components.png', dpi=140, bbox_inches='tight')
    plt.show()
else:
    print('Prophet not available — skipping component plots.')

## 5. XGBoost on Engineered Features

In [ ]:
# ── 5.1 Load feature matrix for XGBoost ──────────────────────────────────
try:
    feat_cols = joblib.load(MODELS_DIR / 'feature_cols.pkl')['price']
except FileNotFoundError:
    feat_cols = [
        'trend_weeks','sin_w52_k1','cos_w52_k1','sin_w52_k2','cos_w52_k2',
        'sin_month','cos_month','is_dry_season','is_harvest_q4',
        'lag_1w','lag_4w','lag_8w','lag_12w','lag_52w',
        'roll_mean_4w','roll_mean_12w','roll_std_12w','momentum_4w',
    ]

try:
    price_feat = pd.read_parquet(FEAT_DIR / 'price_features.parquet')
    price_feat['date'] = pd.to_datetime(price_feat['date'])
    crop_feat = (
        price_feat[price_feat.crop == DEMO_CROP]
        .set_index('date')
        .sort_index()
    )
    avail_feats = [c for c in feat_cols if c in crop_feat.columns]
    X_xgb = crop_feat[avail_feats].dropna()
    y_xgb = crop_feat.loc[X_xgb.index, 'price_xaf_per_kg']
    print(f'Feature matrix: {X_xgb.shape}')
except FileNotFoundError:
    print('Feature parquet not found — using raw lags as proxy.')
    df_c = price_df[price_df.crop == DEMO_CROP].set_index('date').sort_index()
    for lag in [1,2,4,8,12,26,52]:
        df_c[f'lag_{lag}'] = df_c['price_xaf_per_kg'].shift(lag)
    df_c = df_c.dropna()
    lag_feats = [f'lag_{l}' for l in [1,2,4,8,12,26,52]]
    X_xgb = df_c[lag_feats]; y_xgb = df_c['price_xaf_per_kg']; avail_feats = lag_feats

In [ ]:
# ── 5.2 Time-series cross-validation ─────────────────────────────────────
if XGB_AVAILABLE:
    tscv = TimeSeriesSplit(n_splits=5, test_size=TEST_SIZE)
    xgb_cv_scores = []

    for fold, (tr_idx, te_idx) in enumerate(tscv.split(X_xgb)):
        X_tr, X_te = X_xgb.iloc[tr_idx], X_xgb.iloc[te_idx]
        y_tr, y_te = y_xgb.iloc[tr_idx], y_xgb.iloc[te_idx]

        xgb_model = xgb.XGBRegressor(
            n_estimators=300, learning_rate=0.05,
            max_depth=5, subsample=0.8, colsample_bytree=0.8,
            random_state=42, n_jobs=-1, verbosity=0,
        )
        xgb_model.fit(X_tr, y_tr,
                      eval_set=[(X_te, y_te)],
                      early_stopping_rounds=30, verbose=False)
        preds  = xgb_model.predict(X_te)
        rmse   = np.sqrt(mean_squared_error(y_te, preds))
        mape   = np.mean(np.abs((y_te - preds) / (y_te + 1e-9))) * 100
        xgb_cv_scores.append({'fold':fold+1,'rmse':rmse,'mape':mape})
        print(f'  Fold {fold+1}: RMSE={rmse:.2f}  MAPE={mape:.2f}%')

    # Final model on last train split
    tr_idx_f, te_idx_f = list(tscv.split(X_xgb))[-1]
    X_tr_f, X_te_f = X_xgb.iloc[tr_idx_f], X_xgb.iloc[te_idx_f]
    y_tr_f, y_te_f = y_xgb.iloc[tr_idx_f], y_xgb.iloc[te_idx_f]

    xgb_final = xgb.XGBRegressor(
        n_estimators=400, learning_rate=0.04,
        max_depth=5, subsample=0.8, colsample_bytree=0.8,
        random_state=42, n_jobs=-1, verbosity=0,
    )
    xgb_final.fit(X_tr_f, y_tr_f, verbose=False)
    xgb_preds  = xgb_final.predict(X_te_f)
    xgb_metrics= evaluate_forecast(y_te_f.values, xgb_preds, 'XGBoost (feature-based)')

    cv_df = pd.DataFrame(xgb_cv_scores)
    print(f'\nCV Mean RMSE: {cv_df.rmse.mean():.2f} ± {cv_df.rmse.std():.2f}')
    print(f'CV Mean MAPE: {cv_df.mape.mean():.2f}% ± {cv_df.mape.std():.2f}%')
else:
    print('XGBoost unavailable.')
    xgb_preds   = sarimax_preds.copy()
    xgb_metrics = arima_metrics

In [ ]:
# ── 5.3 XGBoost feature importance ───────────────────────────────────────
if XGB_AVAILABLE:
    fi = pd.Series(xgb_final.feature_importances_, index=avail_feats).sort_values(ascending=True)

    fig, ax = plt.subplots(figsize=(9, max(5, len(fi)*0.38)))
    colors = [LIME if v > fi.median() else '#444444' for v in fi.values]
    ax.barh(fi.index, fi.values, color=colors, edgecolor='none', height=0.65)
    ax.set_title(f'XGBoost Feature Importance — {DEMO_CROP} Price')
    ax.set_xlabel('Importance')
    plt.tight_layout()
    plt.savefig(OUTPUT_DIR/'xgb_feature_importance.png', dpi=140, bbox_inches='tight')
    plt.show()

## 6. Ensemble Blending (Premium Tier)

In [ ]:
# ── 6.1 Weighted ensemble — weights inversely proportional to MAPE ────────
def inverse_mape_weights(*metrics_list):
    """Returns weights inversely proportional to each model's MAPE."""
    mapes = np.array([m.get('mape', 10.0) for m in metrics_list])
    mapes = np.maximum(mapes, 0.1)   # avoid division by zero
    inv   = 1.0 / mapes
    return inv / inv.sum()

# Align predictions to common test indices
arima_arr   = sarimax_preds[:TEST_SIZE]
prophet_arr = prophet_preds[:TEST_SIZE]

# XGBoost predictions may be on a different slice; align to TEST_SIZE
if XGB_AVAILABLE and len(xgb_preds) >= TEST_SIZE:
    xgb_arr = xgb_preds[-TEST_SIZE:]
else:
    xgb_arr = arima_arr.copy()

weights = inverse_mape_weights(arima_metrics, prophet_metrics,
                                xgb_metrics if XGB_AVAILABLE else arima_metrics)
ensemble_preds = (weights[0] * arima_arr +
                  weights[1] * prophet_arr +
                  weights[2] * xgb_arr)

ensemble_metrics = evaluate_forecast(test_series.values[:TEST_SIZE], ensemble_preds, 'Ensemble (Premium tier)')

print(f'\nEnsemble weights — ARIMA:{weights[0]:.3f}  Prophet:{weights[1]:.3f}  XGB:{weights[2]:.3f}')

## 7. Backtesting & Evaluation

In [ ]:
# ── 7.1 Forecast vs actuals plot ──────────────────────────────────────────
test_idx = test_series.index[:TEST_SIZE]
actual   = test_series.values[:TEST_SIZE]

fig, axes = plt.subplots(2, 1, figsize=(16, 10), sharex=True)

# Top: all model forecasts
ax = axes[0]
ax.plot(test_idx, actual,        color='white',   lw=2,   label='Actual',   zorder=4)
ax.plot(test_idx, arima_arr,     color='#38B6FF', lw=1.5, label='ARIMA (Free)',   linestyle='--')
ax.plot(test_idx, prophet_arr,   color='#FFB800', lw=1.5, label='Prophet (Medium)', linestyle='-.')
ax.plot(test_idx, ensemble_preds,color=LIME,      lw=2,   label='Ensemble (Premium)', zorder=3)

# Confidence band from Prophet
ax.fill_between(test_idx, prophet_lower[:TEST_SIZE], prophet_upper[:TEST_SIZE],
                alpha=0.12, color='#FFB800', label='Prophet 80% CI')

ax.set_title(f'{DEMO_CROP.upper()} Price Forecast vs Actuals (XAF/kg)')
ax.set_ylabel('Price (XAF/kg)'); ax.legend(fontsize=9, ncol=3); ax.grid(True, alpha=0.25)

# Bottom: residuals
ax2 = axes[1]
ax2.plot(test_idx, actual - ensemble_preds, color=LIME, lw=1.5, label='Ensemble residuals')
ax2.plot(test_idx, actual - arima_arr,      color='#38B6FF', lw=1, alpha=0.7, label='ARIMA residuals')
ax2.axhline(0, color='#555555', lw=1, linestyle='--')
ax2.set_title('Forecast Residuals')
ax2.set_ylabel('Error (XAF/kg)'); ax2.legend(fontsize=9); ax2.grid(True, alpha=0.25)

plt.tight_layout()
plt.savefig(OUTPUT_DIR/'forecast_vs_actuals.png', dpi=140, bbox_inches='tight')
plt.show()

In [ ]:
# ── 7.2 Rolling-window MAPE across horizons ───────────────────────────────
def mape_at_horizon(actual, predicted, h):
    """MAPE for the h-step ahead forecast."""
    return np.mean(np.abs((actual[h:] - predicted[h:]) / (actual[h:] + 1e-9))) * 100

horizons  = [1, 2, 4, 8, 12]
mape_arima    = [mape_at_horizon(actual, arima_arr, h-1)    for h in horizons]
mape_prophet  = [mape_at_horizon(actual, prophet_arr, h-1)  for h in horizons]
mape_ensemble = [mape_at_horizon(actual, ensemble_preds,h-1) for h in horizons]

fig, ax = plt.subplots(figsize=(10, 5))
ax.plot(horizons, mape_arima,    marker='o', color='#38B6FF', lw=2, label='ARIMA (Free)')
ax.plot(horizons, mape_prophet,  marker='s', color='#FFB800', lw=2, label='Prophet (Medium)')
ax.plot(horizons, mape_ensemble, marker='^', color=LIME,      lw=2.5, label='Ensemble (Premium)')
ax.set_xlabel('Forecast Horizon (weeks)'); ax.set_ylabel('MAPE (%)')
ax.set_title('MAPE Degradation by Forecast Horizon')
ax.legend(); ax.grid(True, alpha=0.3)
ax.set_xticks(horizons)
plt.tight_layout()
plt.savefig(OUTPUT_DIR/'mape_by_horizon.png', dpi=140, bbox_inches='tight')
plt.show()

## 8. Tier Comparison Summary

In [ ]:
# ── 8.1 Comparison table ──────────────────────────────────────────────────
comparison = pd.DataFrame([
    {'Tier':'Free',    'Model':'ARIMA/SARIMAX',           **arima_metrics},
    {'Tier':'Medium',  'Model':'Prophet',                  **prophet_metrics},
    {'Tier':'Premium', 'Model':'Ensemble (ARIMA+Prophet+XGB)', **ensemble_metrics},
])
print('=== MODEL COMPARISON ===')
print(comparison.to_string(index=False))

fig, axes = plt.subplots(1, 3, figsize=(14, 5))
tier_colors = ['#444444','#FFB800',LIME]
for ax, metric, ylabel in zip(axes,['rmse','mape','bias'],['RMSE','MAPE (%)','Forecast Bias']):
    bars = ax.bar(comparison['Tier'], comparison[metric], color=tier_colors, edgecolor='none', width=0.5)
    ax.set_title(ylabel); ax.set_ylabel(ylabel)
    for bar, v in zip(bars, comparison[metric]):
        ax.text(bar.get_x()+bar.get_width()/2, bar.get_height()+0.001,
                f'{v:.3f}', ha='center', va='bottom', fontsize=10)

fig.suptitle('Forecast Model Performance by Subscription Tier', fontsize=13)
plt.tight_layout()
plt.savefig(OUTPUT_DIR/'tier_comparison.png', dpi=140, bbox_inches='tight')
plt.show()

## 9. Save Models

In [ ]:
# ── 9.1 Save all forecasting models ───────────────────────────────────────
# SARIMAX
sarimax_fit.save(str(MODELS_DIR / f'sarimax_{DEMO_CROP}.pkl'))
print(f'SARIMAX saved: models/sarimax_{DEMO_CROP}.pkl')

# Prophet
if PROPHET_AVAILABLE:
    joblib.dump(prophet_model, MODELS_DIR / f'prophet_{DEMO_CROP}.pkl')
    print(f'Prophet  saved: models/prophet_{DEMO_CROP}.pkl')

# XGBoost
if XGB_AVAILABLE:
    joblib.dump(xgb_final, MODELS_DIR / f'xgb_price_{DEMO_CROP}.pkl')
    print(f'XGBoost  saved: models/xgb_price_{DEMO_CROP}.pkl')

# Ensemble config
joblib.dump({
    'weights':         weights.tolist(),
    'arima_metrics':   arima_metrics,
    'prophet_metrics': prophet_metrics,
    'ensemble_metrics':ensemble_metrics,
    'demo_crop':       DEMO_CROP,
    'feature_cols':    avail_feats,
}, MODELS_DIR / 'ensemble_config.pkl')
print('Ensemble config saved: models/ensemble_config.pkl')

print('\n=== All forecasting models saved. ===')

In [ ]:
# ── 9.2 Notebook summary ──────────────────────────────────────────────────
print('=' * 60)
print('  PRICE FORECASTING — NOTEBOOK SUMMARY')
print('=' * 60)
print(f'  Crop modelled   : {DEMO_CROP}')
print(f'  Train / Test    : {len(train_series)} / {TEST_SIZE} weeks')
print(f'  Forecast horizon: {FORECAST_HORIZON} weeks')
print()
print('  Model Performance:')
for _, row in comparison.iterrows():
    print(f"    [{row['Tier']:8s}] {row['Model']:35s}  MAPE={row['mape']:.2f}%")
print()
print('  Next steps:')
print('  - Run this notebook for all 14 crops')
print('  - Add exogenous regressors (exchange rate, inflation, import price)')
print('  - Hyperparameter tune XGBoost with Optuna')
print('=' * 60)